# Análisis de Audio con Basic-Pitch (Spotify)
Extrae notas, pitch y genera un archivo MIDI a partir de cualquier grabación de audio.  
Ideal para analizar quenas, pincuyos y traversas.

In [2]:
# Verificar que basic-pitch está instalado correctamente
import importlib.metadata
try:
    version = importlib.metadata.version("basic-pitch")
    print(f"✓ basic-pitch versión: {version}")
except importlib.metadata.PackageNotFoundError:
    print("⚠ basic-pitch no está instalado en este kernel.")
    print("  Ejecuta: pip install basic-pitch")


✓ basic-pitch versión: 0.4.0


## 1. Importaciones

In [1]:
import os
import numpy as np
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH
import pretty_midi

## 2. Configuración del archivo de audio
Cambia `AUDIO_FILE` por la ruta a tu grabación (`.wav`, `.mp3`, `.ogg`, `.flac`).

In [ ]:
# =====================================================================
# CONFIGURA AQUÍ TU ARCHIVO DE AUDIO
# =====================================================================
AUDIO_FILE = "grabacion_quena.wav"   # <-- cambia por tu archivo

# Parámetros de detección (ajustables)
ONSET_THRESHOLD = 0.5    # sensibilidad para detectar inicio de nota (0-1)
FRAME_THRESHOLD = 0.3    # sensibilidad por frame (0-1)
MIN_NOTE_LEN    = 127    # duración mínima de nota en ms
MIN_FREQ        = 200.0  # frecuencia mínima en Hz (filtra ruido bajo)
MAX_FREQ        = 2000.0 # frecuencia máxima en Hz

# Verificar que el archivo existe
if not os.path.exists(AUDIO_FILE):
    print(f"⚠ Archivo '{AUDIO_FILE}' no encontrado.")
    print("  Coloca tu grabación en la carpeta del proyecto y actualiza AUDIO_FILE.")
else:
    print(f"✓ Archivo encontrado: {AUDIO_FILE}")

## 3. Análisis con Basic-Pitch

In [ ]:
if os.path.exists(AUDIO_FILE):
    print("Analizando audio... (puede tardar unos segundos)\n")

    model_output, midi_data, note_events = predict(
        AUDIO_FILE,
        onset_threshold=ONSET_THRESHOLD,
        frame_threshold=FRAME_THRESHOLD,
        minimum_note_length=MIN_NOTE_LEN,
        minimum_frequency=MIN_FREQ,
        maximum_frequency=MAX_FREQ,
    )

    print(f"✓ Análisis completado. Notas detectadas: {len(note_events)}\n")
    print(f"{'#':<5} {'Inicio(s)':<12} {'Fin(s)':<10} {'MIDI':<8} {'Freq(Hz)':<12} {'Vel.':<6}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        print(f"{i+1:<5} {start:<12.3f} {end:<10.3f} {pitch:<8} {freq:<12.1f} {vel:.2f}")
else:
    print("⚠ Saltando análisis: archivo de audio no disponible.")
    note_events = []
    midi_data = None

## 4. Convertir notas MIDI a nombres musicales

In [ ]:
NOTAS_ES = ["Do", "Do#", "Re", "Re#", "Mi", "Fa",
            "Fa#", "Sol", "Sol#", "La", "La#", "Si"]

def midi_a_nombre(midi_num):
    nombre = NOTAS_ES[midi_num % 12]
    octava = (midi_num // 12) - 1
    return f"{nombre}{octava}"

if note_events:
    print(f"\n{'#':<5} {'Nota':<8} {'Inicio(s)':<12} {'Duración(ms)':<15} {'Freq(Hz)':<12}")
    print("-" * 55)
    for i, (start, end, pitch, vel, _) in enumerate(note_events):
        freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
        dur_ms = (end - start) * 1000
        print(f"{i+1:<5} {midi_a_nombre(int(pitch)):<8} {start:<12.3f} {dur_ms:<15.0f} {freq:<12.1f}")
else:
    print("Sin notas detectadas.")

## 5. Guardar el resultado como archivo MIDI

In [ ]:
if midi_data is not None:
    OUTPUT_MIDI = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.mid"
    midi_data.write(OUTPUT_MIDI)
    print(f"✓ MIDI guardado en: {OUTPUT_MIDI}")
    print("  Puedes abrirlo en MuseScore, GarageBand o importarlo con music21.")
else:
    print("Sin datos MIDI para guardar.")

## 6. Estadísticas de la grabación

In [ ]:
if note_events:
    from collections import Counter

    pitches   = [int(n[2]) for n in note_events]
    freqs     = [440.0 * (2.0 ** ((p - 69) / 12.0)) for p in pitches]
    durations = [(n[1] - n[0]) * 1000 for n in note_events]
    conteo    = Counter([midi_a_nombre(p) for p in pitches])

    print("=== ESTADÍSTICAS ===")
    print(f"  Total de notas        : {len(note_events)}")
    print(f"  Nota más grave        : {midi_a_nombre(min(pitches))}  ({min(freqs):.1f} Hz)")
    print(f"  Nota más aguda        : {midi_a_nombre(max(pitches))}  ({max(freqs):.1f} Hz)")
    print(f"  Duración media (ms)   : {np.mean(durations):.0f}")
    print(f"  Duración mínima (ms)  : {np.min(durations):.0f}")
    print(f"  Duración máxima (ms)  : {np.max(durations):.0f}")
    print(f"\n  Notas más frecuentes:")
    for nota, cnt in conteo.most_common(8):
        barra = "█" * cnt
        print(f"    {nota:<6} {barra}  ({cnt})")
else:
    print("Sin datos para estadísticas.")

## 7. Exportar lista de notas a CSV

In [ ]:
import csv

if note_events:
    OUTPUT_CSV = AUDIO_FILE.rsplit(".", 1)[0] + "_notas.csv"
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["#", "Nota", "MIDI", "Freq_Hz",
                         "Inicio_s", "Fin_s", "Duracion_ms", "Velocidad"])
        for i, (start, end, pitch, vel, _) in enumerate(note_events):
            freq   = 440.0 * (2.0 ** ((pitch - 69) / 12.0))
            dur_ms = (end - start) * 1000
            writer.writerow([i+1, midi_a_nombre(int(pitch)), int(pitch),
                             round(freq, 2), round(start, 3), round(end, 3),
                             round(dur_ms, 0), round(vel, 3)])
    print(f"✓ CSV guardado en: {OUTPUT_CSV}")
else:
    print("Sin notas para exportar.")